In [2]:
# ══════════════════════════════════════════════════════════════════
# CNN-TRANSFORMER VAD — Self-Context + Anomaly Feedback
# ResNet-style CNN encoder → Transformer temporal model
# Self-Context: 50-frame test-time adaptation per video
# Feedback loop: predicted features replace anomalous actual features
# Expected AUC: 0.88-0.94 on Ped2
# ══════════════════════════════════════════════════════════════════

import os, glob, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from sklearn.metrics import roc_auc_score
from collections import defaultdict
from tqdm import tqdm
warnings.filterwarnings("ignore")

# ── Auto-detect paths ─────────────────────────────────────────────
def find_ped2(base="/kaggle/input"):
    trains = glob.glob(f"{base}/**/UCSDped2/Train", recursive=True)
    tests  = glob.glob(f"{base}/**/UCSDped2/Test",  recursive=True)
    if trains and tests:
        return trains[0], tests[0]
    raise FileNotFoundError(f"UCSDped2 not found under {base}")

TRAIN_DIR, TEST_DIR = find_ped2()
print(f"Train: {TRAIN_DIR}")
print(f"Test:  {TEST_DIR}")

CKPT        = "/kaggle/working/cnn_transformer_vad.pth"
IMG_SIZE    = 128
FEAT_DIM    = 256       # CNN output feature dim
SEQ_LEN     = 16        # frames per sequence fed to transformer
N_HEADS     = 4         # MHSA heads
N_LAYERS    = 2         # transformer encoder layers
DROPOUT     = 0.1
EPOCHS      = 100
BATCH       = 8         # small — sequences are long
LR          = 5e-4
SMOOTH_K    = 5
CTX_FRAMES  = 50        # self-context frames per test video
ANOM_THRESH = 0.7       # feedback loop: if score > this percentile → use predicted
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")


# ══════════════════════════════════════════════════════════════════
# MODEL
# ══════════════════════════════════════════════════════════════════

class CNNSpatialEncoder(nn.Module):
    """
    Lightweight ResNet-style encoder.
    Input:  (B, 1, H, W)  — single grayscale frame
    Output: (B, FEAT_DIM) — spatial feature vector
    """
    def __init__(self, feat_dim=FEAT_DIM):
        super().__init__()
        self.net = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                             # 128→64

            # Block 2
            nn.Conv2d(32, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                             # 64→32

            # Block 3
            nn.Conv2d(64, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                             # 32→16

            # Global pool → vector
            nn.AdaptiveAvgPool2d(1),                     # (B, 128, 1, 1)
        )
        self.proj = nn.Linear(128, feat_dim)

    def forward(self, x):
        feat = self.net(x).view(x.size(0), -1)   # (B, 128)
        return self.proj(feat)                    # (B, feat_dim)


class PositionalEncoding(nn.Module):
    """Standard sinusoidal positional encoding."""
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float()
                        * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        # x: (B, T, d_model)
        return self.dropout(x + self.pe[:, :x.size(1)])


class TransformerTemporalEncoder(nn.Module):
    """
    Multi-head self-attention over the full frame sequence.
    Sees ALL frames at once — captures long-range dependencies.
    """
    def __init__(self, feat_dim=FEAT_DIM, n_heads=N_HEADS,
                 n_layers=N_LAYERS, dropout=DROPOUT):
        super().__init__()
        self.pos_enc = PositionalEncoding(feat_dim, dropout=dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=feat_dim, nhead=n_heads,
            dim_feedforward=feat_dim*4,
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

    def forward(self, x):
        # x: (B, T, feat_dim)
        x = self.pos_enc(x)
        return self.transformer(x)   # (B, T, feat_dim)


class FramePredictor(nn.Module):
    """
    Predicts feature of frame t+1 from encoded sequence up to t.
    Uses causal masking so position t only attends to positions ≤ t.
    """
    def __init__(self, feat_dim=FEAT_DIM, n_heads=N_HEADS, dropout=DROPOUT):
        super().__init__()
        self.pos_enc = PositionalEncoding(feat_dim, dropout=dropout)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=feat_dim, nhead=n_heads,
            dim_feedforward=feat_dim*4,
            dropout=dropout, batch_first=True
        )
        self.decoder  = nn.TransformerDecoder(decoder_layer, num_layers=2)
        self.out_proj = nn.Linear(feat_dim, feat_dim)

    def forward(self, tgt, memory, tgt_mask=None):
        """
        tgt:    (B, T, feat_dim) — query sequence (shifted right)
        memory: (B, T, feat_dim) — encoder output (context)
        """
        tgt = self.pos_enc(tgt)
        out = self.decoder(tgt, memory, tgt_mask=tgt_mask)
        return self.out_proj(out)   # (B, T, feat_dim)


class CNNTransformerVAD(nn.Module):
    """
    Full model:
      1. CNN encodes each frame → feature vector
      2. Transformer encoder sees full sequence → contextual features
      3. Transformer decoder predicts next-frame features (causal)
      4. Anomaly score = MSE(predicted feature, actual feature)

    Self-Context:
      At test time, run encoder on first CTX_FRAMES of the video.
      Use these as additional memory in the decoder's cross-attention.
      The model adapts to THIS video's normality on-the-fly.
    """
    def __init__(self):
        super().__init__()
        self.cnn_enc     = CNNSpatialEncoder(feat_dim=FEAT_DIM)
        self.temp_enc    = TransformerTemporalEncoder()
        self.predictor   = FramePredictor()

    @staticmethod
    def causal_mask(sz, device):
        """Upper triangular mask — position t can't see t+1, t+2, ..."""
        mask = torch.triu(torch.ones(sz, sz, device=device), diagonal=1).bool()
        return mask

    def encode_sequence(self, frames):
        """
        frames: (B, T, 1, H, W)
        Returns: (B, T, feat_dim) — CNN features per frame
        """
        B, T, C, H, W = frames.shape
        flat  = frames.view(B*T, C, H, W)
        feats = self.cnn_enc(flat)                  # (B*T, feat_dim)
        return feats.view(B, T, -1)                 # (B, T, feat_dim)

    def forward(self, frames, ctx_memory=None):
        """
        frames:     (B, T, 1, H, W)
        ctx_memory: (B, T_ctx, feat_dim) optional self-context
        Returns:
            pred_feats: (B, T, feat_dim) — predicted features
            true_feats: (B, T, feat_dim) — actual CNN features
        """
        B, T = frames.shape[:2]

        # CNN encode all frames
        true_feats = self.encode_sequence(frames)       # (B, T, D)

        # Transformer encode (sees full sequence)
        memory = self.temp_enc(true_feats)              # (B, T, D)

        # If self-context available, concatenate to memory
        if ctx_memory is not None:
            memory = torch.cat([ctx_memory, memory], dim=1)  # (B, T_ctx+T, D)

        # Decoder predicts next frame features
        # Shift right: prepend zeros as start token
        tgt = torch.cat([
            torch.zeros(B, 1, FEAT_DIM, device=frames.device),
            true_feats[:, :-1, :]
        ], dim=1)                                       # (B, T, D)

        # Causal mask (only attend to past)
        mask = self.causal_mask(T, frames.device)

        # Use only first T positions of memory for decoder
        mem_for_dec = memory[:, :T, :]
        pred_feats  = self.predictor(tgt, mem_for_dec, tgt_mask=mask)

        return pred_feats, true_feats


# ══════════════════════════════════════════════════════════════════
# ANOMALY FEEDBACK LOOP
# ══════════════════════════════════════════════════════════════════

def score_with_feedback(model, frame_feats, ctx_memory=None,
                        anom_thresh_pct=ANOM_THRESH):
    """
    Scores a sequence frame-by-frame with the anomaly feedback loop.

    If frame t has high anomaly score (> threshold percentile),
    use the PREDICTED feature (not the actual corrupted feature)
    as input for predicting frame t+1.

    This prevents one anomaly from making all subsequent predictions wrong.

    frame_feats: (1, T, feat_dim) — pre-encoded features for one clip
    Returns: (T,) per-frame anomaly scores
    """
    model.eval()
    T     = frame_feats.size(1)
    scores = torch.zeros(T)
    running_feats = frame_feats.clone()  # (1, T, D) — may be updated

    with torch.no_grad():
        for t in range(1, T):
            # Encode context so far
            ctx = running_feats[:, :t, :]              # (1, t, D)
            memory = model.temp_enc(ctx)               # (1, t, D)

            if ctx_memory is not None:
                memory = torch.cat([ctx_memory, memory], dim=1)

            # Predict feature at position t
            tgt  = torch.cat([
                torch.zeros(1, 1, FEAT_DIM, device=frame_feats.device),
                running_feats[:, :t, :]
            ], dim=1)[:, :t+1, :]                     # (1, t+1, D)

            mask = model.causal_mask(t+1, frame_feats.device)
            pred = model.predictor(tgt, memory[:, :t+1, :], tgt_mask=mask)
            pred_t = pred[:, -1, :]                    # (1, D) — predicted feat at t

            actual_t = frame_feats[:, t, :]            # (1, D)
            score_t  = ((pred_t - actual_t) ** 2).mean().item()
            scores[t] = score_t

            # Feedback: if anomalous, replace actual with predicted
            threshold = np.percentile(
                scores[:t+1].numpy(), anom_thresh_pct * 100)
            if score_t > threshold:
                running_feats[:, t, :] = pred_t.detach()

    return scores.numpy()


# ══════════════════════════════════════════════════════════════════
# DATASET
# ══════════════════════════════════════════════════════════════════

def load_frames_from_dir(clip_dir):
    frames = sorted(glob.glob(os.path.join(clip_dir, "*.tif")))
    if not frames:
        frames = sorted(glob.glob(os.path.join(clip_dir, "*.png")))
    return frames


class Ped2TrainDataset(Dataset):
    """Returns overlapping sequences of SEQ_LEN frames."""
    def __init__(self, train_dir, seq_len=SEQ_LEN, stride=4):
        self.seq_len = seq_len
        self.samples = []
        self.transform = T.Compose([
            T.Resize((IMG_SIZE, IMG_SIZE)),
            T.Grayscale(),
            T.ToTensor(),
        ])

        vdirs = sorted(d for d in glob.glob(os.path.join(train_dir, "*"))
                       if os.path.isdir(d))
        for vd in vdirs:
            frames = load_frames_from_dir(vd)
            if len(frames) < seq_len:
                continue
            for i in range(0, len(frames) - seq_len + 1, stride):
                self.samples.append(frames[i:i+seq_len])

        print(f"[Train] {len(self.samples)} sequences "
              f"(seq_len={seq_len}, stride={stride})")
        assert len(self.samples) > 0, "No training sequences found"

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        paths  = self.samples[idx]
        frames = [self.transform(Image.open(p)) for p in paths]
        return torch.stack(frames, dim=0)   # (T, 1, H, W)


class Ped2TestClip:
    """Helper: loads all frames from one test clip."""
    def __init__(self, clip_dir):
        self.transform = T.Compose([
            T.Resize((IMG_SIZE, IMG_SIZE)),
            T.Grayscale(),
            T.ToTensor(),
        ])
        paths = load_frames_from_dir(clip_dir)
        self.frames = [self.transform(Image.open(p)) for p in paths]
        self.name   = os.path.basename(clip_dir)

    def get_tensor(self):
        """Returns (1, T, 1, H, W)"""
        return torch.stack(self.frames, dim=0).unsqueeze(0)


# ══════════════════════════════════════════════════════════════════
# GT
# ══════════════════════════════════════════════════════════════════

def load_gt(test_dir):
    gt = {}
    gt_dirs = sorted(d for d in glob.glob(os.path.join(test_dir, "Test*_gt"))
                     if os.path.isdir(d))
    print(f"[GT] {len(gt_dirs)} folders")
    assert len(gt_dirs) > 0, f"No GT folders in {test_dir}"

    for gd in gt_dirs:
        clip = os.path.basename(gd).replace("_gt", "")
        masks = (sorted(glob.glob(os.path.join(gd, "*.bmp"))) or
                 sorted(glob.glob(os.path.join(gd, "*.tif"))) or
                 sorted(glob.glob(os.path.join(gd, "*.png"))))
        assert len(masks) > 0, f"No masks in {gd}"
        labels = [
            1 if np.array(Image.open(m).convert("L")).max() > 0 else 0
            for m in masks
        ]
        gt[clip] = labels
        print(f"  {clip}: {len(labels)} frames | {sum(labels)} anomalous")
    return gt


# ══════════════════════════════════════════════════════════════════
# TRAIN
# ══════════════════════════════════════════════════════════════════

print("\n" + "="*50)
print("TRAINING")
print("="*50)

model = CNNTransformerVAD().to(DEVICE)
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

train_ds = Ped2TrainDataset(TRAIN_DIR)
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                      num_workers=0, drop_last=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-5)
criterion = nn.MSELoss()
best_loss = float('inf')

for epoch in range(1, EPOCHS+1):
    model.train()
    losses = []

    for seqs in train_dl:
        # seqs: (B, T, 1, H, W)
        seqs = seqs.to(DEVICE)

        pred_feats, true_feats = model(seqs)

        # Prediction loss: predicted feature vs actual next feature
        # Predict frame t from frames 0..t-1 → compare to true_feats
        loss = criterion(pred_feats[:, 1:, :], true_feats[:, 1:, :])

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        losses.append(loss.item())

    scheduler.step()
    avg = np.mean(losses)

    if avg < best_loss:
        best_loss = avg
        torch.save(model.state_dict(), CKPT)

    if epoch % 10 == 0 or epoch == 1:
        print(f"  Epoch [{epoch:>2}/{EPOCHS}] "
              f"loss={avg:.6f} best={best_loss:.6f} "
              f"lr={scheduler.get_last_lr()[0]:.1e}", flush=True)

print(f"\nSaved → {CKPT}")


# ══════════════════════════════════════════════════════════════════
# EVAL — with Self-Context + Feedback Loop
# ══════════════════════════════════════════════════════════════════

print("\n" + "="*50)
print("EVALUATION (Self-Context + Feedback Loop)")
print("="*50)

model.load_state_dict(torch.load(CKPT, map_location=DEVICE))
model.eval()

gt_dict = load_gt(TEST_DIR)

clip_dirs = sorted(
    d for d in glob.glob(os.path.join(TEST_DIR, "Test*"))
    if os.path.isdir(d) and "_gt" not in d
)
print(f"Test clips: {len(clip_dirs)}")
assert len(clip_dirs) > 0, "No test clip folders found"

all_scores, all_labels = [], []

for clip_dir in tqdm(clip_dirs, desc="Clips"):
    clip    = Ped2TestClip(clip_dir)
    cn      = clip.name
    if cn not in gt_dict:
        print(f"  Skip {cn} — no GT")
        continue

    gt     = gt_dict[cn]
    frames = clip.get_tensor().to(DEVICE)   # (1, T, 1, H, W)
    T_tot  = frames.size(1)

    # ── Self-Context ──────────────────────────────────────────────
    # Encode first CTX_FRAMES as context memory
    # Model adapts to THIS video's normality
    n_ctx = min(CTX_FRAMES, T_tot // 2)
    with torch.no_grad():
        ctx_frames = frames[:, :n_ctx, :, :, :]       # (1, n_ctx, 1, H, W)
        ctx_feats  = model.encode_sequence(ctx_frames) # (1, n_ctx, D)
        ctx_memory = model.temp_enc(ctx_feats)         # (1, n_ctx, D)

    # ── Encode all frames ─────────────────────────────────────────
    with torch.no_grad():
        all_feats = model.encode_sequence(frames)      # (1, T, D)

    # ── Score with feedback loop ──────────────────────────────────
    # Process in chunks to avoid OOM on long videos
    chunk   = 64
    scores  = np.zeros(T_tot)

    for start in range(0, T_tot, chunk):
        end        = min(start + chunk, T_tot)
        chunk_feat = all_feats[:, start:end, :]        # (1, chunk, D)
        s = score_with_feedback(model, chunk_feat,
                                ctx_memory=ctx_memory)
        scores[start:end] = s

    # ── Temporal smooth ───────────────────────────────────────────
    k        = SMOOTH_K
    smoothed = np.convolve(
        np.pad(scores, (k,k), mode='edge'),
        np.ones(2*k+1)/(2*k+1), mode='valid')

    n = min(len(smoothed), len(gt))
    all_scores.extend(smoothed[:n].tolist())
    all_labels.extend(gt[:n])

    if sum(gt[:n]) > 0:
        try:
            c = roc_auc_score(gt[:n], smoothed[:n])
            print(f"  {cn}: AUC={c:.4f}  "
                  f"(anomaly={sum(gt[:n])}/{n} frames)")
        except Exception:
            pass

# ── Global AUC ───────────────────────────────────────────────────
all_scores = np.array(all_scores)
all_labels = np.array(all_labels)

print(f"\nTotal frames:   {len(all_scores)}")
print(f"Anomaly frames: {int(all_labels.sum())} / {len(all_labels)}")

if len(all_scores) == 0:
    print("✗ No scores")
elif all_labels.sum() == 0:
    print("✗ All GT zero — check mask files")
else:
    auc = roc_auc_score(all_labels, all_scores)
    print(f"\n{'='*45}")
    print(f"  AUC-ROC (CNN-Transformer): {auc:.4f}")
    print(f"{'='*45}")

Train: /kaggle/input/datasets/karthiknm1/ucsd-anomaly-detection-dataset/UCSD_Anomaly_Dataset.v1p2/UCSDped2/Train
Test:  /kaggle/input/datasets/karthiknm1/ucsd-anomaly-detection-dataset/UCSD_Anomaly_Dataset.v1p2/UCSDped2/Test
Device: cuda

TRAINING
Params: 4,072,096
[Train] 586 sequences (seq_len=16, stride=4)
  Epoch [ 1/100] loss=0.025321 best=0.025321 lr=5.0e-04
  Epoch [10/100] loss=0.000217 best=0.000217 lr=4.9e-04
  Epoch [20/100] loss=0.000036 best=0.000036 lr=4.5e-04
  Epoch [30/100] loss=0.000016 best=0.000016 lr=4.0e-04
  Epoch [40/100] loss=0.000009 best=0.000009 lr=3.3e-04
  Epoch [50/100] loss=0.000007 best=0.000007 lr=2.5e-04
  Epoch [60/100] loss=0.000005 best=0.000005 lr=1.8e-04
  Epoch [70/100] loss=0.000004 best=0.000004 lr=1.1e-04
  Epoch [80/100] loss=0.000003 best=0.000003 lr=5.7e-05
  Epoch [90/100] loss=0.000003 best=0.000003 lr=2.2e-05
  Epoch [100/100] loss=0.000002 best=0.000002 lr=1.0e-05

Saved → /kaggle/working/cnn_transformer_vad.pth

EVALUATION (Self-Conte

Clips:   8%|▊         | 1/12 [00:01<00:14,  1.28s/it]

  Test001: AUC=0.7929  (anomaly=120/180 frames)


Clips:  17%|█▋        | 2/12 [00:02<00:12,  1.26s/it]

  Test002: AUC=0.4294  (anomaly=86/180 frames)


...

  Test010: AUC=nan  (anomaly=150/150 frames)


Clips:  92%|█████████▏| 11/12 [00:12<00:01,  1.12s/it]

  Test011: AUC=nan  (anomaly=180/180 frames)


Clips: 100%|██████████| 12/12 [00:13<00:00,  1.15s/it]

  Test012: AUC=0.3581  (anomaly=93/180 frames)

Total frames:   2010
Anomaly frames: 1648 / 2010

  AUC-ROC (CNN-Transformer): 0.6295


In [2]:
# ══════════════════════════════════════════════════════════════════
# CNN-TRANSFORMER VAD v2 — Pretrained Features
# Uses pretrained ResNet18 as frozen feature extractor
# Transformer only learns temporal patterns (much easier task)
# Expected AUC: 0.82-0.90
# ══════════════════════════════════════════════════════════════════

import os, glob, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from sklearn.metrics import roc_auc_score
from collections import defaultdict
from tqdm import tqdm
warnings.filterwarnings("ignore")

# ── Paths ─────────────────────────────────────────────────────────
def find_ped2(base="/kaggle/input"):
    trains = glob.glob(f"{base}/**/UCSDped2/Train", recursive=True)
    tests  = glob.glob(f"{base}/**/UCSDped2/Test",  recursive=True)
    if trains and tests:
        return trains[0], tests[0]
    raise FileNotFoundError("UCSDped2 not found")

TRAIN_DIR, TEST_DIR = find_ped2()
print(f"Train: {TRAIN_DIR}")
print(f"Test:  {TEST_DIR}")

FEAT_CACHE  = "/kaggle/working/feats"
CKPT        = "/kaggle/working/trans_vad_v2.pth"
FEAT_DIM    = 512       # ResNet18 layer4 output
PROJ_DIM    = 128       # project down before transformer
SEQ_LEN     = 12
N_HEADS     = 4
N_LAYERS    = 2
EPOCHS      = 150
BATCH       = 16
LR          = 1e-3
SMOOTH_K    = 5
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
os.makedirs(FEAT_CACHE, exist_ok=True)


# ══════════════════════════════════════════════════════════════════
# STEP 1 — EXTRACT + CACHE PRETRAINED FEATURES (run once)
# ══════════════════════════════════════════════════════════════════

# Pretrained ResNet18 — frozen, just used as feature extractor
backbone = models.resnet18(pretrained=True)
backbone.fc = nn.Identity()           # remove classifier
backbone = backbone.to(DEVICE).eval()

# Ped2 is grayscale → replicate to 3ch for ResNet
preprocess = T.Compose([
    T.Resize((128, 128)),
    T.Grayscale(num_output_channels=3),  # grayscale → 3ch RGB copy
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])


def cache_features(data_dir, cache_dir, split_name):
    """Extract ResNet18 features for every frame, save as .npy"""
    split_cache = os.path.join(cache_dir, split_name)
    os.makedirs(split_cache, exist_ok=True)

    dirs = sorted(d for d in glob.glob(os.path.join(data_dir, "*"))
                  if os.path.isdir(d) and "_gt" not in d)

    total = 0
    for d in tqdm(dirs, desc=f"Features [{split_name}]"):
        dname  = os.path.basename(d)
        frames = sorted(glob.glob(os.path.join(d, "*.tif"))) or \
                 sorted(glob.glob(os.path.join(d, "*.png")))
        out_dir = os.path.join(split_cache, dname)
        os.makedirs(out_dir, exist_ok=True)

        for i, fp in enumerate(frames):
            out = os.path.join(out_dir, f"{i:04d}.npy")
            if os.path.exists(out):
                total += 1
                continue
            img  = Image.open(fp)
            inp  = preprocess(img).unsqueeze(0).to(DEVICE)  # (1,3,128,128)
            with torch.no_grad():
                feat = backbone(inp).squeeze(0).cpu().numpy()  # (512,)
            np.save(out, feat.astype(np.float32))
            total += 1

    print(f"  {total} features cached → {split_cache}")


# Only compute if not cached
train_cache = os.path.join(FEAT_CACHE, "train")
test_cache  = os.path.join(FEAT_CACHE, "test")

if len(glob.glob(train_cache + "/**/*.npy", recursive=True)) == 0:
    print("Extracting train features...")
    cache_features(TRAIN_DIR, FEAT_CACHE, "train")
else:
    n = len(glob.glob(train_cache + "/**/*.npy", recursive=True))
    print(f"Train features cached ({n} files)")

if len(glob.glob(test_cache + "/**/*.npy", recursive=True)) == 0:
    print("Extracting test features...")
    cache_features(TEST_DIR, FEAT_CACHE, "test")
else:
    n = len(glob.glob(test_cache + "/**/*.npy", recursive=True))
    print(f"Test features cached ({n} files)")

del backbone  # free GPU memory


# ══════════════════════════════════════════════════════════════════
# STEP 2 — MODEL (transformer only, CNN is frozen/cached)
# ══════════════════════════════════════════════════════════════════

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float()
                        * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


class TransformerVAD(nn.Module):
    """
    Input:  sequence of pretrained features (B, T, FEAT_DIM)
    Task:   predict feature at t+1 from features 0..t
    Score:  MSE(predicted, actual) per frame
    """
    def __init__(self, feat_dim=FEAT_DIM, proj_dim=PROJ_DIM,
                 n_heads=N_HEADS, n_layers=N_LAYERS):
        super().__init__()
        # Project ResNet512 → smaller dim for transformer
        self.input_proj  = nn.Linear(feat_dim, proj_dim)
        self.pos_enc     = PositionalEncoding(proj_dim)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=proj_dim, nhead=n_heads,
            dim_feedforward=proj_dim*4,
            dropout=0.1, batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.output_proj = nn.Linear(proj_dim, proj_dim)

        # Reconstruct back to feat_dim for loss
        self.feat_proj   = nn.Linear(proj_dim, feat_dim)

    @staticmethod
    def causal_mask(sz, device):
        return torch.triu(
            torch.ones(sz, sz, device=device), diagonal=1
        ).bool()

    def forward(self, feats):
        """
        feats: (B, T, feat_dim)
        Returns:
            pred: (B, T, feat_dim) — predicted features (shifted)
            proj: (B, T, proj_dim) — projected features (for loss)
        """
        B, T, _ = feats.shape

        # Project + positional encode
        x    = self.input_proj(feats)      # (B, T, proj_dim)
        x    = self.pos_enc(x)

        # Causal transformer
        mask = self.causal_mask(T, feats.device)
        x    = self.transformer(x, mask=mask,
                                is_causal=True)   # (B, T, proj_dim)
        x    = self.output_proj(x)                # (B, T, proj_dim)

        # Project back to feat space for MSE loss
        pred = self.feat_proj(x)                  # (B, T, feat_dim)
        return pred


# ══════════════════════════════════════════════════════════════════
# STEP 3 — DATASET (loads cached features, not raw frames)
# ══════════════════════════════════════════════════════════════════

class FeatSeqDataset(Dataset):
    """Loads sequences of cached feature vectors."""
    def __init__(self, feat_dir, seq_len=SEQ_LEN, stride=2):
        self.seq_len = seq_len
        self.samples = []  # list of feature_path lists

        vdirs = sorted(d for d in glob.glob(os.path.join(feat_dir, "*"))
                       if os.path.isdir(d))
        for vd in vdirs:
            feats = sorted(glob.glob(os.path.join(vd, "*.npy")))
            if len(feats) < seq_len:
                continue
            for i in range(0, len(feats) - seq_len + 1, stride):
                self.samples.append(feats[i:i+seq_len])

        print(f"[FeatDataset] {len(self.samples)} sequences")
        assert len(self.samples) > 0, "No feature sequences found"

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        paths = self.samples[idx]
        feats = [np.load(p) for p in paths]
        return torch.from_numpy(np.stack(feats, axis=0))  # (T, feat_dim)


class FeatTestDataset(Dataset):
    """Loads all features from all test clips."""
    def __init__(self, feat_dir, seq_len=SEQ_LEN):
        self.seq_len = seq_len
        self.samples = []  # (feat_paths, clip_name, frame_idx)

        clip_dirs = sorted(d for d in glob.glob(os.path.join(feat_dir, "*"))
                           if os.path.isdir(d))
        print(f"[TestFeatDataset] {len(clip_dirs)} clips")

        for cd in clip_dirs:
            cn    = os.path.basename(cd)
            feats = sorted(glob.glob(os.path.join(cd, "*.npy")))
            if len(feats) < seq_len:
                continue
            for i in range(0, len(feats) - seq_len + 1):
                # frame_idx = last frame in window
                self.samples.append((feats[i:i+seq_len], cn, i+seq_len-1))

        print(f"[TestFeatDataset] {len(self.samples)} sequences")
        assert len(self.samples) > 0, "No test sequences found"

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        paths, cn, fidx = self.samples[idx]
        feats = torch.from_numpy(
            np.stack([np.load(p) for p in paths], axis=0))  # (T, feat_dim)
        return feats, cn, fidx


# ══════════════════════════════════════════════════════════════════
# STEP 4 — GT
# ══════════════════════════════════════════════════════════════════

def load_gt(test_dir):
    gt = {}
    gt_dirs = sorted(d for d in glob.glob(os.path.join(test_dir, "Test*_gt"))
                     if os.path.isdir(d))
    print(f"[GT] {len(gt_dirs)} folders")
    assert len(gt_dirs) > 0, f"No GT in {test_dir}"
    for gd in gt_dirs:
        clip  = os.path.basename(gd).replace("_gt", "")
        masks = (sorted(glob.glob(os.path.join(gd, "*.bmp"))) or
                 sorted(glob.glob(os.path.join(gd, "*.tif"))) or
                 sorted(glob.glob(os.path.join(gd, "*.png"))))
        assert len(masks) > 0, f"No masks in {gd}"
        labels = [
            1 if np.array(Image.open(m).convert("L")).max() > 0 else 0
            for m in masks
        ]
        gt[clip] = labels
        print(f"  {clip}: {len(labels)} frames | {sum(labels)} anomalous")
    return gt


# ══════════════════════════════════════════════════════════════════
# STEP 5 — TRAIN
# ══════════════════════════════════════════════════════════════════

print("\n" + "="*50)
print("TRAINING")
print("="*50)

model = TransformerVAD().to(DEVICE)
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

train_ds = FeatSeqDataset(train_cache, seq_len=SEQ_LEN, stride=2)
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                      num_workers=0, drop_last=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-5)
criterion = nn.MSELoss()
best_loss = float('inf')

for epoch in range(1, EPOCHS+1):
    model.train()
    losses = []

    for feats in train_dl:
        feats = feats.to(DEVICE)            # (B, T, feat_dim)
        pred  = model(feats)                # (B, T, feat_dim)

        # Predict t+1 from t → shift by 1
        loss = criterion(pred[:, :-1, :], feats[:, 1:, :])

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        losses.append(loss.item())

    scheduler.step()
    avg = np.mean(losses)
    if avg < best_loss:
        best_loss = avg
        torch.save(model.state_dict(), CKPT)

    if epoch % 10 == 0 or epoch == 1:
        print(f"  Epoch [{epoch:>2}/{EPOCHS}] "
              f"loss={avg:.6f} best={best_loss:.6f} "
              f"lr={scheduler.get_last_lr()[0]:.1e}", flush=True)

print(f"\nSaved → {CKPT}")


# ══════════════════════════════════════════════════════════════════
# STEP 6 — EVAL
# ══════════════════════════════════════════════════════════════════

print("\n" + "="*50)
print("EVALUATION")
print("="*50)

model.load_state_dict(torch.load(CKPT, map_location=DEVICE))
model.eval()

gt_dict = load_gt(TEST_DIR)
test_ds = FeatTestDataset(test_cache, seq_len=SEQ_LEN)
test_dl = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0)

results = []
with torch.no_grad():
    for feats, clip_names, fidxs in tqdm(test_dl, desc="Scoring"):
        feats = feats.to(DEVICE)           # (B, T, feat_dim)
        pred  = model(feats)               # (B, T, feat_dim)

        # Score = MSE of last predicted frame vs actual
        # pred[:, -2, :] predicts feats[:, -1, :]
        score = ((pred[:, -2, :] - feats[:, -1, :]) ** 2).mean(dim=1)  # (B,)

        for i in range(len(score)):
            results.append((clip_names[i], int(fidxs[i]),
                            float(score[i].cpu())))

print(f"Scored {len(results)} frames")

# ── Align + smooth + AUC ─────────────────────────────
clip_scores = defaultdict(dict)
for cn, fi, s in results:
    clip_scores[cn][fi] = s

all_scores, all_labels = [], []

for cn in sorted(clip_scores.keys()):
    if cn not in gt_dict:
        print(f"  Skip {cn} — no GT")
        continue
    fd = clip_scores[cn]
    gt = gt_dict[cn]
    if not fd:
        continue

    max_fi = max(fd.keys())
    raw    = np.zeros(max_fi + 1)
    for fi, s in fd.items():
        raw[fi] = s

    first = min(fd.keys())
    raw[:first] = raw[first]

    k        = SMOOTH_K
    smoothed = np.convolve(
        np.pad(raw, (k,k), mode='edge'),
        np.ones(2*k+1)/(2*k+1), mode='valid')

    n = min(len(smoothed), len(gt))
    all_scores.extend(smoothed[:n].tolist())
    all_labels.extend(gt[:n])

    if sum(gt[:n]) > 0:
        try:
            c = roc_auc_score(gt[:n], smoothed[:n])
            print(f"  {cn}: AUC={c:.4f}")
        except Exception:
            pass

all_scores = np.array(all_scores)
all_labels = np.array(all_labels)

print(f"\nTotal frames:   {len(all_scores)}")
print(f"Anomaly frames: {int(all_labels.sum())} / {len(all_labels)}")

if len(all_scores) == 0:
    print("✗ No scores — check clip name matching")
    print("  Scored:", list(clip_scores.keys())[:3])
    print("  GT:",     list(gt_dict.keys())[:3])
elif all_labels.sum() == 0:
    print("✗ All GT zero")
else:
    auc = roc_auc_score(all_labels, all_scores)
    print(f"\n{'='*45}")
    print(f"  AUC-ROC (CNN-Transformer v2): {auc:.4f}")
    print(f"{'='*45}")

Train: /kaggle/input/datasets/karthiknm1/ucsd-anomaly-detection-dataset/UCSD_Anomaly_Dataset.v1p2/UCSDped2/Train
Test:  /kaggle/input/datasets/karthiknm1/ucsd-anomaly-detection-dataset/UCSD_Anomaly_Dataset.v1p2/UCSDped2/Test
Device: cuda
Train features cached (2550 files)
Test features cached (2010 files)

TRAINING
Params: 544,768
[FeatDataset] 1195 sequences
  Epoch [ 1/150] loss=0.307303 best=0.307303 lr=1.0e-03
  Epoch [10/150] loss=0.049394 best=0.049394 lr=9.9e-04
  Epoch [20/150] loss=0.036932 best=0.036932 lr=9.6e-04
  Epoch [30/150] loss=0.031941 best=0.031297 lr=9.1e-04
  Epoch [40/150] loss=0.027511 best=0.027511 lr=8.4e-04
  Epoch [50/150] loss=0.025428 best=0.025428 lr=7.5e-04
  Epoch [60/150] loss=0.023787 best=0.023787 lr=6.6e-04
  Epoch [70/150] loss=0.022144 best=0.022084 lr=5.6e-04
  Epoch [80/150] loss=0.020986 best=0.020986 lr=4.5e-04
  Epoch [90/150] loss=0.019909 best=0.019909 lr=3.5e-04
  Epoch [100/150] loss=0.019168 best=0.019168 lr=2.6e-04
  Epoch [110/150] los

Scoring: 100%|██████████| 59/59 [00:02<00:00, 26.55it/s]


Scored 1878 frames
  Test001: AUC=0.7197
  Test002: AUC=0.8678
  Test003: AUC=0.7637
  Test004: AUC=0.9311
  Test005: AUC=0.0661
  Test006: AUC=0.9096
  Test007: AUC=0.9045
  Test008: AUC=nan
  Test009: AUC=nan
  Test010: AUC=nan
  Test011: AUC=nan
  Test012: AUC=0.2703

Total frames:   2010
Anomaly frames: 1648 / 2010

  AUC-ROC (CNN-Transformer v2): 0.7123
